# 💳 Credit Card Fraud Detection — Exploratory Data Analysis

**Dataset:** Kaggle Credit Card Fraud Detection  
**Rows:** 284,807 transactions  
**Target:** `Class` — 0 = Legitimate, 1 = Fraud  
**Features:** Time, V1–V28 (PCA-transformed anonymous), Amount

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'axes.grid':        True,
    'font.family':      'DejaVu Sans',
})

FRAUD_COLOR  = '#f85149'   # red
LEGIT_COLOR  = '#3fb950'   # green
ACCENT_COLOR = '#ff7b24'   # amber
NEUTRAL      = '#58a6ff'   # blue

DATA_PATH = Path('..') / 'data' / 'creditcard.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print('\n=== DATA TYPES ===')
print(df.dtypes)
print('\n=== NULL VALUES ===')
print(df.isnull().sum().sum(), 'total nulls')
print('\n=== BASIC STATS ===')
df.describe()

## 2. Class Distribution — The Imbalance Problem

In [ ]:
counts = df['Class'].value_counts()
labels = ['Legitimate (0)', 'Fraud (1)']
colors = [LEGIT_COLOR, FRAUD_COLOR]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Bar chart
ax = axes[0]
bars = ax.bar(labels, counts.values, color=colors, width=0.5, edgecolor='#30363d', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold',
            color='#e6edf3')
ax.set_title('Transaction Count by Class', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Transactions')
ax.set_yscale('log')
ax.set_xlabel('Class')

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.3f%%', startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2},
    textprops={'color': '#e6edf3', 'fontsize': 11}
)
for at in autotexts: at.set_fontsize(10)
ax2.set_title('Class Proportion', fontsize=14, fontweight='bold', pad=15)

plt.suptitle('⚠️  Severe Class Imbalance: Only 0.17% Fraud', fontsize=15,
             fontweight='bold', color=FRAUD_COLOR, y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Legitimate: {counts[0]:,}  |  Fraud: {counts[1]:,}  |  Ratio: {counts[0]/counts[1]:.0f}:1')

## 3. Transaction Amount Distribution by Class

In [ ]:
legit = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Boxplot
ax = axes[0]
bp = ax.boxplot(
    [legit.clip(upper=legit.quantile(0.99)), fraud],
    labels=['Legitimate', 'Fraud'],
    patch_artist=True,
    notch=True,
    boxprops=dict(linewidth=1.5),
    medianprops=dict(linewidth=2.5, color='#e6edf3'),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
)
bp['boxes'][0].set_facecolor(LEGIT_COLOR + '40')
bp['boxes'][0].set_edgecolor(LEGIT_COLOR)
bp['boxes'][1].set_facecolor(FRAUD_COLOR + '40')
bp['boxes'][1].set_edgecolor(FRAUD_COLOR)
ax.set_title('Amount Distribution (99th pct cap)', fontsize=13, fontweight='bold')
ax.set_ylabel('Transaction Amount ($)')

# KDE
ax2 = axes[1]
legit.clip(upper=legit.quantile(0.99)).plot.kde(ax=ax2, color=LEGIT_COLOR, linewidth=2, label='Legitimate')
fraud.plot.kde(ax=ax2, color=FRAUD_COLOR, linewidth=2, label='Fraud')
ax2.set_title('Amount Density (KDE)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Transaction Amount ($)')
ax2.set_xlim(-100, legit.quantile(0.99))
ax2.legend()

plt.suptitle('Transaction Amount: Legitimate vs Fraud', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f'Legit   — Median: ${legit.median():.2f}  |  Mean: ${legit.mean():.2f}  |  Max: ${legit.max():.2f}')
print(f'Fraud   — Median: ${fraud.median():.2f}  |  Mean: ${fraud.mean():.2f}  |  Max: ${fraud.max():.2f}')

## 4. Time Distribution — When Do Frauds Happen?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Convert seconds to hours
legit_hrs = df[df['Class'] == 0]['Time'] / 3600
fraud_hrs = df[df['Class'] == 1]['Time'] / 3600

legit_hrs.plot.kde(ax=ax, color=LEGIT_COLOR, linewidth=2.5, label='Legitimate', alpha=0.85)
fraud_hrs.plot.kde(ax=ax, color=FRAUD_COLOR, linewidth=2.5, label='Fraud', alpha=0.85)

ax.fill_between(ax.lines[0].get_xdata(), ax.lines[0].get_ydata(), alpha=0.12, color=LEGIT_COLOR)
ax.fill_between(ax.lines[1].get_xdata(), ax.lines[1].get_ydata(), alpha=0.18, color=FRAUD_COLOR)

ax.set_title('Transaction Time Distribution — Legitimate vs Fraud', fontsize=14, fontweight='bold')
ax.set_xlabel('Time (hours since first transaction)')
ax.set_ylabel('Density')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('time_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(18, 14))
fig.patch.set_facecolor('#0d1117')

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

cmap = sns.diverging_palette(10, 145, as_cmap=True)
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap=cmap, center=0, vmin=-1, vmax=1,
    annot=False, linewidths=0.3, linecolor='#0d1117',
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'}
)
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 6. Top Features — Correlation with Class (Target)

In [ ]:
corr_with_target = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0d1117')

colors = [FRAUD_COLOR if v < 0 else LEGIT_COLOR for v in corr_with_target.values]
bars = ax.barh(corr_with_target.index, corr_with_target.values, color=colors,
               edgecolor='#30363d', linewidth=0.6)
ax.axvline(0, color='#e6edf3', linewidth=0.8, linestyle='--')
ax.set_title('Feature Correlation with Class (Fraud=1)', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, corr_with_target.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=8, color='#e6edf3')

plt.tight_layout()
plt.savefig('feature_correlation_target.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('Top 5 features most correlated with fraud:')
print(corr_with_target.head())

## 7. V-Feature Distributions: Fraud vs Legitimate (Top 12)

In [ ]:
top_features = corr_with_target.head(12).index.tolist()
legit_df = df[df['Class'] == 0]
fraud_df = df[df['Class'] == 1]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    legit_df[feat].plot.kde(ax=ax, color=LEGIT_COLOR, linewidth=2, label='Legit', alpha=0.85)
    fraud_df[feat].plot.kde(ax=ax, color=FRAUD_COLOR, linewidth=2, label='Fraud', alpha=0.85)
    ax.fill_between(ax.lines[0].get_xdata(), ax.lines[0].get_ydata(), alpha=0.1, color=LEGIT_COLOR)
    ax.fill_between(ax.lines[1].get_xdata(), ax.lines[1].get_ydata(), alpha=0.2, color=FRAUD_COLOR)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Top 12 Feature Distributions — Fraud vs Legitimate', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 8. Transaction Amount — Log Scale Violin

In [ ]:
df_plot = df.copy()
df_plot['Amount_log'] = np.log1p(df_plot['Amount'])
df_plot['Class_label'] = df_plot['Class'].map({0: 'Legitimate', 1: 'Fraud'})

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0d1117')

palette = {'Legitimate': LEGIT_COLOR, 'Fraud': FRAUD_COLOR}
sns.violinplot(data=df_plot, x='Class_label', y='Amount_log',
               palette=palette, inner='box', ax=ax, linewidth=1.2)

ax.set_title('Transaction Amount (log scale) — Legitimate vs Fraud', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('log(1 + Amount)')
plt.tight_layout()
plt.savefig('amount_violin.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 9. SMOTE Preview — Before vs After Balancing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

X = df.drop('Class', axis=1)
y = df['Class']

X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

before_counts = y_train.value_counts()

smote = SMOTE(random_state=42)
_, y_resampled = smote.fit_resample(X_train, y_train)
after_counts = pd.Series(y_resampled).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#0d1117')

for ax, counts, title in zip(axes,
                              [before_counts, after_counts],
                              ['Before SMOTE', 'After SMOTE']):
    ax.bar(['Legitimate', 'Fraud'], [counts[0], counts[1]],
           color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor='#30363d', linewidth=0.8)
    for i, (label, val) in enumerate(zip(['Legitimate', 'Fraud'], [counts[0], counts[1]])):
        ax.text(i, val + 300, f'{val:,}', ha='center', fontsize=11, fontweight='bold', color='#e6edf3')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('SMOTE Oversampling Effect on Training Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Before — Legit: {before_counts[0]:,}  |  Fraud: {before_counts[1]:,}')
print(f'After  — Legit: {after_counts[0]:,}  |  Fraud: {after_counts[1]:,}')

## 10. Quick Model Comparison (Sample)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

# Use a small sample for quick EDA comparison
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_s)
X_test_sc = scaler.transform(X_test_s)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_sc, y_train_s)

# Take 10% for quick EDA only
idx = np.random.RandomState(42).choice(len(X_res), int(0.1 * len(X_res)), replace=False)
X_quick, y_quick = X_res[idx], y_res[idx]

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=50, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=50, verbosity=0, eval_metric='logloss', random_state=42),
}

results = []
for name, clf in classifiers.items():
    clf.fit(X_quick, y_quick)
    y_prob = clf.predict_proba(X_test_sc)[:, 1]
    y_pred = clf.predict(X_test_sc)
    results.append({
        'Model': name,
        'ROC-AUC':   round(roc_auc_score(y_test_s, y_prob), 4),
        'F1':        round(f1_score(y_test_s, y_pred), 4),
        'Precision': round(precision_score(y_test_s, y_pred), 4),
        'Recall':    round(recall_score(y_test_s, y_pred), 4),
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('=== Quick Model Comparison (10% SMOTE sample) ===')
results_df

## 11. Model Performance Visualization

In [ ]:
metrics = ['ROC-AUC', 'F1', 'Precision', 'Recall']
x = np.arange(len(results_df))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')

metric_colors = [NEUTRAL, ACCENT_COLOR, LEGIT_COLOR, FRAUD_COLOR]
for i, (metric, color) in enumerate(zip(metrics, metric_colors)):
    ax.bar(x + i * width, results_df[metric], width, label=metric,
           color=color, alpha=0.85, edgecolor='#0d1117', linewidth=0.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — ROC-AUC, F1, Precision, Recall', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.axhline(0.85, color='#e6edf3', linestyle='--', linewidth=0.8, alpha=0.5, label='Min threshold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 12. ROC Curve — All Models

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')

palette = plt.cm.Set2(np.linspace(0, 1, len(classifiers)))
for (name, clf), color in zip(classifiers.items(), palette):
    y_prob = clf.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_s, y_prob)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, color=color, label=f'{name} (AUC = {roc_auc_val:.4f})')

ax.plot([0,1],[0,1], linestyle='--', color='#8b949e', linewidth=1.2, label='Random Classifier')
ax.fill_between([0,1],[0,1], alpha=0.05, color='#8b949e')
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 13. Precision-Recall Curve (Critical for Imbalanced Data)

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0d1117')

for (name, clf), color in zip(classifiers.items(), palette):
    y_prob = clf.predict_proba(X_test_sc)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test_s, y_prob)
    ap = average_precision_score(y_test_s, y_prob)
    ax.plot(rec, prec, linewidth=2, color=color, label=f'{name} (AP = {ap:.4f})')

baseline = y_test_s.sum() / len(y_test_s)
ax.axhline(baseline, linestyle='--', color='#8b949e', linewidth=1.2,
           label=f'Baseline (prevalence = {baseline:.4f})')

ax.set_title('Precision-Recall Curves — All Models', fontsize=14, fontweight='bold')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 14. Confusion Matrix — Best Model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_model_name = results_df.iloc[0]['Model']
best_clf = classifiers[best_model_name]
y_pred_best = best_clf.predict(X_test_sc)

cm = confusion_matrix(y_test_s, y_pred_best)

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor('#0d1117')

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='RdYlGn')

ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold', pad=12)
for text in ax.texts: text.set_fontsize(14)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Best model: {best_model_name}')
print(f'True Positives  (fraud caught):    {tp}')
print(f'False Negatives (fraud missed):    {fn}')
print(f'False Positives (false alarms):    {fp}')
print(f'True Negatives  (legit correct):   {tn}')

## 15. Feature Importance — Best Tree-Based Model

In [ ]:
# Use XGBoost or RandomForest for feature importance
fi_model = classifiers.get('XGBoost', classifiers.get('Random Forest'))
fi_name  = 'XGBoost' if 'XGBoost' in classifiers else 'Random Forest'

importances = pd.Series(fi_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#0d1117')

grad = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(importances)))
bars = ax.barh(importances.index[::-1], importances.values[::-1],
               color=grad[::-1], edgecolor='#30363d', linewidth=0.5)

ax.set_title(f'Feature Importance — {fi_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')

for bar, val in zip(bars, importances.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8, color='#e6edf3')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 16. Summary & Key Findings

| Finding | Detail |
|---------|--------|
| **Class Imbalance** | 99.83% legitimate, 0.17% fraud — extreme imbalance |
| **Amount** | Fraud transactions tend to be lower value than legitimate ones |
| **Time** | Fraud has slightly different temporal patterns (less uniform) |
| **Key Features** | V14, V12, V10, V11 show strongest separation between classes |
| **SMOTE** | Balances training set to 50/50 without touching test set |
| **Best Metric** | ROC-AUC preferred over accuracy due to extreme imbalance |

---

**Next Steps:** Run `python -m src.pipeline.train_pipeline` to train the full pipeline and save `artifacts/model.pkl` and `artifacts/preprocessor.pkl`.